# Notebook 01 (Participant): Train + Generate

Complete the `TODO` sections to train a lightweight conditional model and generate designs.

Fallback: set `USE_NEAREST_NEIGHBOR_FALLBACK = True` if training is too slow.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'torch', 'torchvision', 'matplotlib', 'pandas']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from engibench.problems.beams2d.v0 import Beams2D

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = Path('workshops/dcc26/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = ARTIFACT_DIR / 'mini_cond_generator.pt'


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

# Build compact train subset to keep runtime stable in workshop
N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)

# Downsample target to reduce model output size and speed up training
designs_t = th.tensor(designs_np).unsqueeze(1)
lowres_t = F.interpolate(designs_t, size=(25, 50), mode='bilinear', align_corners=False).squeeze(1)
targets_np = lowres_t.reshape(N_TRAIN, -1).numpy()

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('lowres target shape:', targets_np.shape)

In [ ]:
# TODO 1: implement MiniCondGenerator
# Suggested architecture:
# Linear(in_dim, 64) -> ReLU -> Linear(64, 128) -> ReLU -> Linear(128, out_dim) -> Sigmoid

class MiniCondGenerator(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # TODO: define self.net
        raise NotImplementedError('Define model layers')

    def forward(self, x):
        # TODO: return forward pass
        raise NotImplementedError('Implement forward pass')


def upsample_to_design(y_flat: th.Tensor) -> th.Tensor:
    low = y_flat.reshape(-1, 1, 25, 50)
    high = F.interpolate(low, size=(50, 100), mode='bilinear', align_corners=False)
    return high.squeeze(1)

# TODO 2: instantiate model/optimizer/loss
# model = ...
# optimizer = ...
# criterion = ...

raise NotImplementedError('Complete TODO 1/2 in this cell')

In [ ]:
TRAIN_FROM_SCRATCH = True
EPOCHS = 8
BATCH_SIZE = 64

if TRAIN_FROM_SCRATCH:
    # TODO 3: implement training loop over DataLoader
    # - forward
    # - MSE loss
    # - backward/update
    # - print epoch loss
    # - save checkpoint to CKPT_PATH
    raise NotImplementedError('Complete TODO 3 training loop')
elif CKPT_PATH.exists():
    ckpt = th.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print('loaded checkpoint from', CKPT_PATH)
else:
    print('No checkpoint found. Use fallback generation cell below.')

In [ ]:
# TODO 4: implement generation path
# - sample N_SAMPLES test conditions
# - if USE_NEAREST_NEIGHBOR_FALLBACK: nearest-neighbor designs from train subset
# - else: model forward + upsample
# - set `gen_designs`, `baseline_designs`, `test_conds`

USE_NEAREST_NEIGHBOR_FALLBACK = False

raise NotImplementedError('Complete TODO 4 generation path')

In [ ]:
# TODO 5: serialize artifacts for Notebook 02
# - generated_designs.npy
# - baseline_designs.npy
# - conditions.json

raise NotImplementedError('Complete TODO 5 artifact export')

In [ ]:
# Quick visual check of generated designs
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f'gen {i}')
plt.tight_layout()
plt.show()

## Next

Continue with **Notebook 02** to validate and evaluate generated designs against baselines.
